# Crashes EDA

Exploration of the `crashes`, `vehicles` and `persons` tables pulled in `data/crash_data/crashes.ipynb`, based on Fei's work in `fei/merge tables.ipynb`.

Everything here queries the local cache, not the API. SODA returns every value as a string, so numeric columns are cast where needed.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from data.base import get_connection

pd.set_option("display.max_columns", None)

In [ ]:
get_connection().sql("DESCRIBE crashes").df()

In [ ]:
get_connection().sql("DESCRIBE vehicles").df()

In [ ]:
get_connection().sql("DESCRIBE persons").df()

### Vehicle travel direction

In [ ]:
travel_direction_pct = get_connection().sql("""
    SELECT travel_direction,
           COUNT(*) AS vehicles,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM vehicles
    GROUP BY travel_direction
    ORDER BY vehicles DESC
""").df()

travel_direction_pct

In [ ]:
# share of vehicles each year whose travel_direction is missing or "Unknown"
direction_quality = get_connection().sql("""
    SELECT CAST(LEFT(crash_date, 4) AS INTEGER) AS year,
           COUNT(*) AS vehicles,
           COUNT(*) FILTER (WHERE travel_direction IS NULL) * 100.0 / COUNT(*) AS missing_pct,
           COUNT(*) FILTER (WHERE LOWER(TRIM(travel_direction)) = 'unknown') * 100.0 / COUNT(*) AS unknown_pct
    FROM vehicles
    GROUP BY year
    ORDER BY year
""").df()

direction_quality["usable_direction_pct"] = (
    100
    - direction_quality["missing_pct"]
    - direction_quality["unknown_pct"]
)

direction_quality.round(2)

### Person table
If a crash is corresponded to more than one person

In [ ]:
get_connection().sql("""
    SELECT COUNT(*) AS rows,
           COUNT(DISTINCT collision_id) AS unique_collisions
    FROM persons
""").df()

In [ ]:
get_connection().sql("""
    SELECT person_type, COUNT(*) AS persons
    FROM persons
    GROUP BY person_type
    ORDER BY persons DESC
""").df()

In [ ]:
get_connection().sql("""
    SELECT person_injury, COUNT(*) AS persons
    FROM persons
    GROUP BY person_injury
    ORDER BY persons DESC
""").df()

```
CRASHES
1 row ≈ 1 collision
location + severity + crash-level contributing factors
            │
            │ collision_id
            ▼
VEHICLES
multiple vehicles / collision
travel_direction
contributing_factor
pre_crash
point_of_impact
            │
            │ collision_id
            ▼
PERSONS
multiple persons / collision
person_type
person_injury
vehicle_id
```

In [ ]:
get_connection().sql("""
    SELECT COUNT(*) FILTER (WHERE collision_id IS NULL) AS missing_collision_id,
           COUNT(*) FILTER (WHERE person_id IS NULL) AS missing_person_id,
           COUNT(*) FILTER (WHERE vehicle_id IS NULL) AS missing_vehicle_id
    FROM persons
""").df()

In [ ]:
get_connection().sql("""
    SELECT person_type,
           COUNT(*) AS total,
           COUNT(*) FILTER (WHERE vehicle_id IS NULL) AS missing,
           ROUND(COUNT(*) FILTER (WHERE vehicle_id IS NULL) * 100.0 / COUNT(*), 2) AS missing_pct
    FROM persons
    GROUP BY person_type
    ORDER BY total DESC
""").df()

### Linking crashes, vehicles and persons (2025)

In [ ]:
YEAR_WHERE = "crash_date >= '2025-01-01' AND crash_date < '2026-01-01'"

linkage_crashes = get_connection().sql(f"""
    SELECT collision_id, crash_date, latitude, longitude,
           number_of_persons_injured, number_of_persons_killed
    FROM crashes
    WHERE {YEAR_WHERE}
""").df()

print("Crash rows:", len(linkage_crashes))
print("Unique collision IDs:", linkage_crashes["collision_id"].nunique())
print("Missing collision IDs:", linkage_crashes["collision_id"].isna().sum())

In [ ]:
linkage_vehicles = get_connection().sql(f"""
    SELECT collision_id, vehicle_id, travel_direction,
           contributing_factor_1, contributing_factor_2,
           pre_crash, point_of_impact
    FROM vehicles
    WHERE {YEAR_WHERE}
""").df()

linkage_vehicles = linkage_vehicles[
    linkage_vehicles["collision_id"].isin(linkage_crashes["collision_id"])
].copy()

print("Vehicle rows matched:", len(linkage_vehicles))
print("Crashes with vehicle records:", linkage_vehicles["collision_id"].nunique())
print("Crashes without vehicle records:",
      len(linkage_crashes) - linkage_vehicles["collision_id"].nunique())

In [ ]:
linkage_persons = get_connection().sql(f"""
    SELECT collision_id, person_id, vehicle_id, person_type,
           person_injury, ped_role, person_age
    FROM persons
    WHERE {YEAR_WHERE}
""").df()

linkage_persons = linkage_persons[
    linkage_persons["collision_id"].isin(linkage_crashes["collision_id"])
].copy()

print("Person rows matched:", len(linkage_persons))
print("Crashes with person records:", linkage_persons["collision_id"].nunique())
print("Crashes without person records:",
      len(linkage_crashes) - linkage_persons["collision_id"].nunique())

how many vehicles are involved in one crash:

In [ ]:
vehicles_per_crash = linkage_vehicles.groupby("collision_id").size()

print(vehicles_per_crash.describe())

print("\nVehicle count distribution:")
print(vehicles_per_crash.value_counts().sort_index().head(10))

In [ ]:
persons_per_crash = linkage_persons.groupby("collision_id").size()

print(persons_per_crash.describe())

print("\nPerson count distribution:")
print(persons_per_crash.value_counts().sort_index().head(10))

In [ ]:
vehicle_summary = (
    linkage_vehicles
    .assign(
        unsafe_speed=lambda df:
            df["contributing_factor_1"].eq("Unsafe Speed")
            | df["contributing_factor_2"].eq("Unsafe Speed"),

        traffic_control_disregarded=lambda df:
            df["contributing_factor_1"].eq("Traffic Control Disregarded")
            | df["contributing_factor_2"].eq("Traffic Control Disregarded")
    )
    .groupby("collision_id")
    .agg(
        n_vehicles=("vehicle_id", "size"),
        any_unsafe_speed=("unsafe_speed", "any"),
        any_traffic_control_disregarded=("traffic_control_disregarded", "any")
    )
    .reset_index()
)

print("Rows:", len(vehicle_summary))
print("Unique collision IDs:", vehicle_summary["collision_id"].nunique())
print("Crashes with Unsafe Speed:", vehicle_summary["any_unsafe_speed"].sum())
print("Crashes with Traffic Control Disregarded:",
      vehicle_summary["any_traffic_control_disregarded"].sum())

vehicle_summary.head(10)

In [ ]:
person_summary = (
    linkage_persons
    .assign(
        injured=lambda df: df["person_injury"].eq("Injured"),
        killed=lambda df: df["person_injury"].eq("Killed"),
        pedestrian=lambda df: df["person_type"].eq("Pedestrian"),
        bicyclist=lambda df: df["person_type"].eq("Bicyclist")
    )
    .groupby("collision_id")
    .agg(
        n_persons=("person_id", "size"),
        n_injured=("injured", "sum"),
        n_killed=("killed", "sum"),
        n_pedestrians=("pedestrian", "sum"),
        n_bicyclists=("bicyclist", "sum")
    )
    .reset_index()
)

print("Rows:", len(person_summary))
print("Unique collision IDs:", person_summary["collision_id"].nunique())

person_summary.head(10)

In [ ]:
crash_master = (
    linkage_crashes
    .merge(vehicle_summary, on="collision_id", how="left")
    .merge(person_summary, on="collision_id", how="left")
)

print("Rows:", len(crash_master))
print("Unique collision IDs:", crash_master["collision_id"].nunique())
print("Duplicate collision IDs:", crash_master["collision_id"].duplicated().sum())

crash_master.head(10)

In [ ]:
crash_master[crash_master["n_persons"].isna()][
    [
        "collision_id",
        "crash_date",
        "number_of_persons_injured",
        "number_of_persons_killed",
        "n_persons"
    ]
]

In [ ]:
vehicle_crash = linkage_vehicles.merge(
    linkage_crashes[
        [
            "collision_id",
            "crash_date",
            "latitude",
            "longitude",
            "number_of_persons_injured",
            "number_of_persons_killed"
        ]
    ],
    on="collision_id",
    how="left",
    suffixes=("_vehicle", "_crash")
)

print("Rows:", len(vehicle_crash))
print("Unique collision IDs:", vehicle_crash["collision_id"].nunique())
print("Unique vehicle IDs:", vehicle_crash["vehicle_id"].nunique())

vehicle_crash.head(10)

### Coordinate quality

In [ ]:
crash_master["latitude_num"] = pd.to_numeric(crash_master["latitude"], errors="coerce")
crash_master["longitude_num"] = pd.to_numeric(crash_master["longitude"], errors="coerce")

print("Total crashes:", len(crash_master))
print("Missing latitude:", crash_master["latitude"].isna().sum())
print("Missing longitude:", crash_master["longitude"].isna().sum())
print(
    "Zero coordinates:",
    ((crash_master["latitude_num"] == 0) | (crash_master["longitude_num"] == 0)).sum()
)

crash_master[["latitude_num", "longitude_num"]].describe()

In [ ]:
crash_master["valid_coordinates"] = (
    crash_master["latitude_num"].notna()
    & crash_master["longitude_num"].notna()
    & (crash_master["latitude_num"] != 0)
    & (crash_master["longitude_num"] != 0)
)

pd.concat(
    [
        crash_master["valid_coordinates"].value_counts().rename("crashes"),
        crash_master["valid_coordinates"].value_counts(normalize=True).mul(100).round(2).rename("pct")
    ],
    axis=1
)

### Do crash-level injury/death counts match the person table?

In [ ]:
consistency_check = crash_master[crash_master["n_persons"].notna()].copy()

consistency_check["injury_match"] = (
    pd.to_numeric(consistency_check["number_of_persons_injured"], errors="coerce")
    == consistency_check["n_injured"]
)
consistency_check["killed_match"] = (
    pd.to_numeric(consistency_check["number_of_persons_killed"], errors="coerce")
    == consistency_check["n_killed"]
)

print("Crashes compared:", len(consistency_check))
print("Injury matches:", consistency_check["injury_match"].sum())
print("Injury mismatches:", (~consistency_check["injury_match"]).sum())
print("Killed matches:", consistency_check["killed_match"].sum())
print("Killed mismatches:", (~consistency_check["killed_match"]).sum())